In [19]:
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers

# Hyperparameters
num_episodes = 100000
max_battery = 100
epsilon = 1.0
epsilon_min = 0.1
epsilon_decay = 0.0001
gamma = 0.99
learning_rate = 0.005
batch_size = 16

class ReplayBuffer:
    def __init__(self, max_size=10000):
        self.buffer = []
        self.max_size = max_size
    
    def store(self, state, action, reward, next_state):
        if len(self.buffer) >= self.max_size:
            self.buffer.pop(0)
        self.buffer.append((state, action, reward, next_state))
    
    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)
    
    def size(self):
        return len(self.buffer)

class QNetwork(tf.keras.Model):
    def __init__(self):
        super(QNetwork, self).__init__()
        self.dense1 = layers.Dense(16, activation='relu')
        self.dense2 = layers.Dense(16, activation='relu')
        self.dense3 = layers.Dense(2)

    def call(self, state):
        x = self.dense1(state)
        x = self.dense2(x)
        return self.dense3(x)

# TensorFlow configuration
tf.keras.backend.set_floatx('float32')
tf.config.run_functions_eagerly(False)
tf.keras.utils.set_random_seed(42)

# Initialize networks
Q_online = QNetwork()
Q_target = QNetwork()
Q_online_optimizer = tf.keras.optimizers.Adam(learning_rate)
Q_online.compile(optimizer=Q_online_optimizer, loss='mse')

@tf.function
def update_q_values(states, actions, rewards, next_states):
    with tf.GradientTape() as tape:
        q_values = Q_online(states)
        q_value = tf.reduce_sum(q_values * tf.one_hot(actions, 2), axis=1)

        next_q_values = Q_target(next_states)
        next_q_value = tf.reduce_max(next_q_values, axis=1)

        target = tf.cast(rewards, tf.float32) + gamma * next_q_value
        loss = tf.reduce_mean(tf.square(target - q_value))

    grads = tape.gradient(loss, Q_online.trainable_variables)
    Q_online_optimizer.apply_gradients(zip(grads, Q_online.trainable_variables))
    return loss

def update_target_network():
    Q_target.set_weights(Q_online.get_weights())

# Simulated environment generation with fixed seed for reproducibility
np.random.seed(42)
electricity_price_per_unit = np.concatenate([
    np.random.randint(1, 6, size=12), 
    np.random.randint(6, 11, size=12)
])
solar_power_generation = np.concatenate([
    np.zeros(6), 
    np.random.randint(1, 11, size=12), 
    np.zeros(6)
])
electricity_demand = np.concatenate([
    np.random.randint(3, 12, size=6), 
    np.random.randint(12, 20, size=12), 
    np.random.randint(3, 12, size=6)
])

replay_buffer = ReplayBuffer()
rewards = []
# Main training loop
for episode in range(num_episodes):
    state = np.array([0, max_battery], dtype=np.float32)
    total_reward = 0

    for hour in range(24):
        # Epsilon-greedy action selection
        if np.random.rand() < epsilon:
            action = np.random.choice([0, 1])
        else:
            q_values = Q_online.predict(state[np.newaxis], verbose=0)[0]
            action = np.argmax(q_values)

        # Action execution
        if action == 0:  # Charge
            reward = -electricity_price_per_unit[hour] * electricity_demand[hour]
            state[1] = min(max_battery, state[1] + solar_power_generation[hour])
        else:  # Discharge
            if state[1] >= electricity_demand[hour]:
                reward = 0
                state[1] -= electricity_demand[hour]
            else:
                reward = -electricity_price_per_unit[hour] * (electricity_demand[hour] - state[1])
                state[1] = 0

        total_reward += reward
        next_state = np.array([hour + 1, state[1]], dtype=np.float32)
        replay_buffer.store(state, action, reward, next_state)
        state = next_state

        # Experience replay
        if replay_buffer.size() >= batch_size:
            minibatch = replay_buffer.sample(batch_size)
            states_batch = tf.convert_to_tensor(
                np.array([item[0] for item in minibatch]), 
                dtype=tf.float32
            )
            actions_batch = tf.convert_to_tensor(
                np.array([item[1] for item in minibatch]), 
                dtype=tf.int32
            )
            rewards_batch = tf.convert_to_tensor(
                np.array([item[2] for item in minibatch]), 
                dtype=tf.float32
            )
            next_states_batch = tf.convert_to_tensor(
                np.array([item[3] for item in minibatch]), 
                dtype=tf.float32
            )
            update_q_values(states_batch, actions_batch, rewards_batch, next_states_batch)

    # Decay epsilon
    epsilon = max(epsilon_min, epsilon - epsilon_decay)
    rewards.append([episode + 1, total_reward])
    # Update target network periodically
    if episode % 2 == 0:
        update_target_network()

    print(f"Episode {episode + 1}/{num_episodes}, Reward: {total_reward}, Epsilon: {epsilon:.4f}")

Episode 1/100000, Reward: -971, Epsilon: 0.9999
Episode 2/100000, Reward: -1166, Epsilon: 0.9998
Episode 3/100000, Reward: -882, Epsilon: 0.9997
Episode 4/100000, Reward: -1161, Epsilon: 0.9996
Episode 5/100000, Reward: -954.0, Epsilon: 0.9995
Episode 6/100000, Reward: -965.0, Epsilon: 0.9994
Episode 7/100000, Reward: -1022.0, Epsilon: 0.9993
Episode 8/100000, Reward: -1000, Epsilon: 0.9992
Episode 9/100000, Reward: -1152.0, Epsilon: 0.9991
Episode 10/100000, Reward: -724.0, Epsilon: 0.9990
Episode 11/100000, Reward: -1035, Epsilon: 0.9989
Episode 12/100000, Reward: -911, Epsilon: 0.9988
Episode 13/100000, Reward: -1178.0, Epsilon: 0.9987
Episode 14/100000, Reward: -1092, Epsilon: 0.9986
Episode 15/100000, Reward: -861.0, Epsilon: 0.9985
Episode 16/100000, Reward: -828.0, Epsilon: 0.9984
Episode 17/100000, Reward: -1020.0, Epsilon: 0.9983
Episode 18/100000, Reward: -959.0, Epsilon: 0.9982
Episode 19/100000, Reward: -800.0, Epsilon: 0.9981
Episode 20/100000, Reward: -1126, Epsilon: 0.99

KeyboardInterrupt: 